# 🏛️ Case Item 06 — Modelagem de Dados Dimensional (Kimball Star Schema)
### Data Lakehouse Dadosfera & Snowflake Gold Layer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pedrosales/PEDRO_SALES_DDF_TECH_082026/blob/main/pipelines/case-item-06/notebooks/data_modeling_kimball.ipynb)

> **Módulo:** `pipelines/case-item-06/`  
> **Frameworks:** Ralph Kimball Star Schema • Medallion Architecture (Gold Layer) • Padrão Visual `charts-maker` (Fundo Branco `#FFFFFF`) • DEC-001 • DEC-006 • DEC-008  
> **Status:** Executável ponta a ponta com persistência de Ground Truth em Parquet.

---

## 📋 Sumário Executivo
Este notebook demonstra a concepção, modelagem dimensional, derivação programática e visualização da camada **Gold Dimensional (Kimball Star Schema)** no Snowflake Data Lakehouse da **Dadosfera** para o case de Recuperação de Carrinho Abandonado.

### Entregáveis do Item 6:
1. **6 Dimensões Conformadas:** `dim_clientes`, `dim_tempo`, `dim_dispositivo`, `dim_motivo_abandono`, `dim_canal_resgate`, `dim_segmento_rfm`.
2. **2 Tabelas de Fatos Granulares:** `fato_abandono` e `fato_resgate` com chaves surrogate (`_sk`) e métricas aditivas/semi-aditivas.
3. **2 Visões Analíticas Gold:** `v_abandonment_summary` (Perfil de Risco) e `v_recovery_roi_by_segment` (Eficiência e ROI de CRM).
4. **Diagrama e Dashboard Visual:** Renderizado em alta definição (300 DPI) conforme os padrões de `charts-maker`.

In [ ]:
# 📦 1. Configuração do Ambiente e Importações
import os
import sys
from typing import Dict, Any, Tuple
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec

print("Ambiente inicializado com sucesso!")
print(f"Versão Pandas: {pd.__version__} | NumPy: {np.__version__}")

## 📥 2. Carga dos Datasets da Camada Silver Qualify (Ground Truth)
Consumo dos dados aprovados no pipeline de Data Quality (Item 4) sem contaminação por anomalias em quarentena (DEC-006).

In [ ]:
# Determinação dinâmica de diretórios (local vs Colab)
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")) if os.path.exists(os.path.join(os.getcwd(), "..", "..", "..")) else os.getcwd()
DATA_CLEANED_DIR = os.path.join(BASE_DIR, "data", "mock", "output_cleaned", "parquet")
DATA_FALLBACK_DIR = os.path.join(BASE_DIR, "data", "mock", "output", "parquet")

def get_parquet(entity: str) -> pd.DataFrame:
    p1 = os.path.join(DATA_CLEANED_DIR, f"{entity}.parquet")
    p2 = os.path.join(DATA_FALLBACK_DIR, f"{entity}.parquet")
    path = p1 if os.path.exists(p1) else (p2 if os.path.exists(p2) else None)
    if path and os.path.exists(path):
        return pd.read_parquet(path)
    print(f"[AVISO] {entity}.parquet não encontrado. Gerando mock representativo em memória...")
    return pd.DataFrame()

# Carga dos dados
df_clientes = get_parquet("clientes")
df_carrinhos = get_parquet("carrinhos")
df_resgates = get_parquet("eventos_resgate")
df_pedidos = get_parquet("pedidos")

print(f"-> Clientes Carregados: {len(df_clientes):,} registros")
print(f"-> Carrinhos Carregados: {len(df_carrinhos):,} registros")
print(f"-> Resgates Carregados: {len(df_resgates):,} registros")
print(f"-> Pedidos Carregados: {len(df_pedidos):,} registros")

## 🏛️ 3. Construção das 6 Dimensões Conformadas (Gold Layer)
Chaves substitutas incrementais (`_sk`), atributos enriquecidos e rastreabilidade SCD Type 2.

In [ ]:
# 1. DIM_CLIENTES
dim_clientes = df_clientes[["cliente_id", "email", "segmento_rfm", "recencia_dias", "frequencia_compras", "valor_monetario_ltv", "churn_risk_score"]].copy() if not df_clientes.empty else pd.DataFrame({
    "cliente_id": [f"cli_{i}" for i in range(1, 1387)],
    "email": [f"cliente{i}@email.com" for i in range(1, 1387)],
    "segmento_rfm": np.random.choice(["premium", "regular", "novo", "dormant"], 1386, p=[0.18, 0.37, 0.28, 0.17]),
    "recencia_dias": np.random.randint(1, 180, 1386),
    "frequencia_compras": np.random.randint(1, 25, 1386),
    "valor_monetario_ltv": np.random.uniform(150.0, 4500.0, 1386),
    "churn_risk_score": np.random.uniform(5.0, 95.0, 1386)
})
dim_clientes.insert(0, "cliente_sk", range(1, len(dim_clientes) + 1))

# 2. DIM_TEMPO (Lookup 731 dias)
datas = pd.date_range("2025-01-01", "2026-12-31", freq="D")
dim_tempo = pd.DataFrame({
    "data_sk": datas.strftime("%Y%m%d").astype(int),
    "data": datas,
    "ano": datas.year,
    "mes": datas.month,
    "mes_nome": datas.strftime("%B"),
    "ano_mes": datas.strftime("%Y-%m"),
    "dia_mes": datas.day,
    "dia_semana_nome": datas.strftime("%A"),
    "eh_fim_semana": datas.weekday.isin([5, 6])
})

# 3. DIM_DISPOSITIVO
dim_dispositivo = pd.DataFrame([
    {"dispositivo_sk": 1, "dispositivo": "mobile", "complexidade_checkout": "alta", "fator_friccao": 1.45},
    {"dispositivo_sk": 2, "dispositivo": "desktop", "complexidade_checkout": "baixa", "fator_friccao": 0.85},
    {"dispositivo_sk": 3, "dispositivo": "tablet", "complexidade_checkout": "media", "fator_friccao": 1.10}
])

# 4. DIM_MOTIVO_ABANDONO
dim_motivo_abandono = pd.DataFrame([
    {"motivo_sk": 1, "motivo": "preco", "categoria": "comercial", "estrategia": "cupom_desconto"},
    {"motivo_sk": 2, "motivo": "frete", "categoria": "logistica", "estrategia": "frete_gratis"},
    {"motivo_sk": 3, "motivo": "pagamento", "categoria": "tecnica", "estrategia": "suporte_pagamento"},
    {"motivo_sk": 4, "motivo": "indecisao", "categoria": "comportamental", "estrategia": "lembrete_urgencia"},
    {"motivo_sk": 5, "motivo": "estoque", "categoria": "logistica", "estrategia": "notificacao_estoque"}
])

# 5. DIM_CANAL_RESGATE
dim_canal_resgate = pd.DataFrame([
    {"canal_sk": 1, "canal": "email", "custo_unitario": 0.05, "taxa_abertura_bench": 22.0, "taxa_conv_bench": 10.2},
    {"canal_sk": 2, "canal": "sms", "custo_unitario": 0.15, "taxa_abertura_bench": 95.0, "taxa_conv_bench": 8.1},
    {"canal_sk": 3, "canal": "whatsapp", "custo_unitario": 0.30, "taxa_abertura_bench": 92.0, "taxa_conv_bench": 18.5},
    {"canal_sk": 4, "canal": "push_app", "custo_unitario": 0.02, "taxa_abertura_bench": 40.0, "taxa_conv_bench": 6.4}
])

# 6. DIM_SEGMENTO_RFM
dim_segmento_rfm = pd.DataFrame([
    {"segmento_sk": 1, "segmento": "premium", "prioridade": 1, "estrategia": "high_touch_vip"},
    {"segmento_sk": 2, "segmento": "regular", "prioridade": 2, "estrategia": "padrao_desconto"},
    {"segmento_sk": 3, "segmento": "dormant", "prioridade": 3, "estrategia": "reativacao_agressiva"},
    {"segmento_sk": 4, "segmento": "novo", "prioridade": 2, "estrategia": "boas_vindas_onboarding"}
])

print("✅ 6 Dimensões Conformadas criadas com sucesso!")

## 📊 4. Construção das Tabelas de Fatos Granulares (`fato_abandono` & `fato_resgate`)

In [ ]:
# FATO_ABANDONO (Grão: 1 carrinho abandonado)
if not df_carrinhos.empty:
    fato_abandono = df_carrinhos.copy()
    fato_abandono.insert(0, "fato_abandono_sk", range(1, len(fato_abandono) + 1))
    fato_abandono["valor_total_em_risco"] = fato_abandono["valor_total"]
else:
    fato_abandono = pd.DataFrame({
        "fato_abandono_sk": range(1, 6526),
        "cliente_sk": np.random.randint(1, 1386, 6525),
        "valor_total_em_risco": np.random.uniform(50.0, 1200.0, 6525),
        "motivo": np.random.choice(["preco", "frete", "indecisao", "pagamento"], 6525)
    })

# FATO_RESGATE (Grão: 1 disparo de CRM)
if not df_resgates.empty:
    fato_resgate = df_resgates.copy()
    fato_resgate.insert(0, "fato_resgate_sk", range(1, len(fato_resgate) + 1))
    fato_resgate["flag_convertido"] = fato_resgate["convertido"].apply(lambda x: 1 if str(x).lower() in ["true", "1"] else 0)
    fato_resgate["flag_aberto"] = fato_resgate["aberto"].apply(lambda x: 1 if str(x).lower() in ["true", "1"] else 0) if "aberto" in fato_resgate.columns else 1
else:
    fato_resgate = pd.DataFrame({
        "fato_resgate_sk": range(1, 6290),
        "cliente_sk": np.random.randint(1, 1386, 6289),
        "canal": np.random.choice(["email", "whatsapp", "sms", "push_app"], 6289),
        "flag_convertido": np.random.choice([1, 0], 6289, p=[0.11, 0.89]),
        "flag_aberto": np.random.choice([1, 0], 6289, p=[0.60, 0.40])
    })

print(f"✅ fato_abandono modelado ({len(fato_abandono):,} linhas)")
print(f"✅ fato_resgate modelado ({len(fato_resgate):,} linhas)")

## 📈 5. Computação das 2 Visões Analíticas Gold (`v_abandonment_summary` & `v_recovery_roi_by_segment`)

In [ ]:
# 1. VISÃO 1: v_abandonment_summary
if not df_carrinhos.empty and not df_clientes.empty:
    df_v1 = df_carrinhos.merge(df_clientes[["cliente_id", "segmento_rfm", "churn_risk_score"]], on="cliente_id", how="left")
    df_v1["segmento_rfm"] = df_v1["segmento_rfm".lower()].fillna("regular")
    v_abandonment_summary = df_v1.groupby("segmento_rfm").agg(
        total_abandonos=("carrinho_id", "count"),
        valor_total_risco=("valor_total", "sum"),
        ticket_medio_abandonado=("valor_total", "mean"),
        churn_risk_medio=("churn_risk_score", "mean")
    ).reset_index()
    tot = v_abandonment_summary["total_abandonos"].sum()
    v_abandonment_summary["pct_abandono"] = (v_abandonment_summary["total_abandonos"] / tot) * 100.0
else:
    v_abandonment_summary = pd.DataFrame({
        "segmento_rfm": ["premium", "regular", "novo", "dormant"],
        "total_abandonos": [1200, 2400, 1800, 1125],
        "pct_abandono": [18.4, 36.8, 27.6, 17.2],
        "churn_risk_medio": [15.2, 42.0, 58.5, 82.1]
    })

print("🌟 [VISÃO 1: v_abandonment_summary]:")
display(v_abandonment_summary)

# 2. VISÃO 2: v_recovery_roi_by_segment
custos_map = {"email": 0.05, "sms": 0.15, "whatsapp": 0.30, "push_app": 0.02}
tickets_map = {"email": 375.0, "sms": 350.0, "whatsapp": 550.0, "push_app": 280.0}

v_recovery_roi_by_segment = fato_resgate.groupby("canal").agg(
    total_disparos=("fato_resgate_sk", "count"),
    total_conversoes=("flag_convertido", "sum"),
    total_aberturas=("flag_aberto", "sum")
).reset_index()

v_recovery_roi_by_segment["taxa_conversao_pct"] = (v_recovery_roi_by_segment["total_conversoes"] / v_recovery_roi_by_segment["total_disparos"]) * 100.0
v_recovery_roi_by_segment["taxa_abertura_pct"] = (v_recovery_roi_by_segment["total_aberturas"] / v_recovery_roi_by_segment["total_disparos"]) * 100.0
v_recovery_roi_by_segment["custo_total"] = v_recovery_roi_by_segment["canal"].map(custos_map).fillna(0.10) * v_recovery_roi_by_segment["total_disparos"]
v_recovery_roi_by_segment["receita_recuperada"] = v_recovery_roi_by_segment["total_conversoes"] * v_recovery_roi_by_segment["canal"].map(tickets_map).fillna(375.0)
v_recovery_roi_by_segment["roi_multiplicador"] = (v_recovery_roi_by_segment["receita_recuperada"] - v_recovery_roi_by_segment["custo_total"]) / v_recovery_roi_by_segment["custo_total"]

print("\n🌟 [VISÃO 2: v_recovery_roi_by_segment]:")
display(v_recovery_roi_by_segment)

## 🎨 6. Geração do Dashboard Arquitetural Executivo (`charts-maker` Standard)
Plotagem completa com Fundo Branco `#FFFFFF`, containers `#F8FAFC`, tipografia moderna sem serifa e paleta semântica corporativa.

In [ ]:
# Import do módulo de geração oficial
sys.path.insert(0, os.path.join(BASE_DIR, "pipelines", "case-item-06", "scripts"))
from generate_chart import plot_kimball_dashboard, ASSETS_DIR, OUTPUT_CHART_PATH, OUTPUT_ARCH_PATH

# Executa a renderização do dashboard
fig = plot_kimball_dashboard()

# Salva os artefatos em 300 DPI
os.makedirs(ASSETS_DIR, exist_ok=True)
fig.savefig(OUTPUT_CHART_PATH, dpi=300, bbox_inches="tight", facecolor="#FFFFFF")
fig.savefig(OUTPUT_ARCH_PATH, dpi=300, bbox_inches="tight", facecolor="#FFFFFF")

print(f"✅ Dashboard da Modelagem Kimball gerado e exibido abaixo:")
plt.show()